In [18]:
import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from pymongo import MongoClient
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.output_parsers import JsonOutputParser
from langchain_mongodb import MongoDBAtlasVectorSearch
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()

True

In [19]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7,
    groq_api_key=os.getenv("GROQ_API_KEY")
)


In [20]:
llm.invoke("How are you ")

AIMessage(content="I'm functioning properly and ready to assist with any questions or tasks you may have.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 39, 'total_tokens': 57, 'completion_time': 0.026834323, 'prompt_time': 0.005362647, 'queue_time': 0.045560479, 'total_time': 0.03219697}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ab04adca7d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--e8fa3052-2068-491b-ba6e-75515164d9fc-0', usage_metadata={'input_tokens': 39, 'output_tokens': 18, 'total_tokens': 57})

In [21]:
prompt = f"""You are a helpful assistant that generates furniture store item data. 
Generate 10 furniture store items as a JSON array. Each record should include the following fields:
- item_id (string): unique identifier
- item_name (string): name of the furniture
- item_description (string): detailed description
- brand (string): brand name
- manufacturer_address (object): street, city, state, postal_code, country
- prices (object): full_price (number), sale_price (number)
- categories (array of strings): furniture categories
- user_reviews (array of objects): review_date (string), rating (number 1-5), comment (string)
- notes (string): additional notes

Ensure variety in the data and realistic values. Return ONLY the JSON array, no markdown formatting.

Example format:
[
  {{
    "item_id": "FUR001",
    "item_name": "Modern Sofa",
    "item_description": "Comfortable 3-seater sofa with soft cushions",
    "brand": "ComfortPlus",
    "manufacturer_address": {{
      "street": "123 Factory Rd",
      "city": "Los Angeles",
      "state": "CA",
      "postal_code": "90001",
      "country": "USA"
    }},
    "prices": {{
      "full_price": 899.99,
      "sale_price": 699.99
    }},
    "categories": ["Living Room", "Sofas", "Modern"],
    "user_reviews": [
      {{
        "review_date": "2024-01-15",
        "rating": 4.5,
        "comment": "Very comfortable and stylish"
      }}
    ],
    "notes": "Available in multiple colors"
  }}
]"""

print("Generating synthetic data...")
    
response = llm.invoke(prompt)

Generating synthetic data...


In [29]:
response

AIMessage(content='[\n  {\n    "item_id": "FUR001",\n    "item_name": "Modern Sofa",\n    "item_description": "Comfortable 3-seater sofa with soft cushions",\n    "brand": "ComfortPlus",\n    "manufacturer_address": {\n      "street": "123 Factory Rd",\n      "city": "Los Angeles",\n      "state": "CA",\n      "postal_code": "90001",\n      "country": "USA"\n    },\n    "prices": {\n      "full_price": 899.99,\n      "sale_price": 699.99\n    },\n    "categories": ["Living Room", "Sofas", "Modern"],\n    "user_reviews": [\n      {\n        "review_date": "2024-01-15",\n        "rating": 4.5,\n        "comment": "Very comfortable and stylish"\n      }\n    ],\n    "notes": "Available in multiple colors"\n  },\n  {\n    "item_id": "FUR002",\n    "item_name": "Industrial Side Table",\n    "item_description": "Reclaimed wood side table with metal legs",\n    "brand": "Industrial Chic",\n    "manufacturer_address": {\n      "street": "456 Industrial Dr",\n      "city": "Brooklyn",\n      "s

In [30]:
for i in response:
    print(i)

('content', '[\n  {\n    "item_id": "FUR001",\n    "item_name": "Modern Sofa",\n    "item_description": "Comfortable 3-seater sofa with soft cushions",\n    "brand": "ComfortPlus",\n    "manufacturer_address": {\n      "street": "123 Factory Rd",\n      "city": "Los Angeles",\n      "state": "CA",\n      "postal_code": "90001",\n      "country": "USA"\n    },\n    "prices": {\n      "full_price": 899.99,\n      "sale_price": 699.99\n    },\n    "categories": ["Living Room", "Sofas", "Modern"],\n    "user_reviews": [\n      {\n        "review_date": "2024-01-15",\n        "rating": 4.5,\n        "comment": "Very comfortable and stylish"\n      }\n    ],\n    "notes": "Available in multiple colors"\n  },\n  {\n    "item_id": "FUR002",\n    "item_name": "Industrial Side Table",\n    "item_description": "Reclaimed wood side table with metal legs",\n    "brand": "Industrial Chic",\n    "manufacturer_address": {\n      "street": "456 Industrial Dr",\n      "city": "Brooklyn",\n      "state":

In [38]:
import json

data = json.loads(response.content)
print(data[0]['item_name'])  # Example: "Modern Sofa"


Modern Sofa


In [43]:
for i in data:
    print(i)

{'item_id': 'FUR001', 'item_name': 'Modern Sofa', 'item_description': 'Comfortable 3-seater sofa with soft cushions', 'brand': 'ComfortPlus', 'manufacturer_address': {'street': '123 Factory Rd', 'city': 'Los Angeles', 'state': 'CA', 'postal_code': '90001', 'country': 'USA'}, 'prices': {'full_price': 899.99, 'sale_price': 699.99}, 'categories': ['Living Room', 'Sofas', 'Modern'], 'user_reviews': [{'review_date': '2024-01-15', 'rating': 4.5, 'comment': 'Very comfortable and stylish'}], 'notes': 'Available in multiple colors'}
{'item_id': 'FUR002', 'item_name': 'Industrial Side Table', 'item_description': 'Reclaimed wood side table with metal legs', 'brand': 'Industrial Chic', 'manufacturer_address': {'street': '456 Industrial Dr', 'city': 'Brooklyn', 'state': 'NY', 'postal_code': '11201', 'country': 'USA'}, 'prices': {'full_price': 249.99, 'sale_price': 199.99}, 'categories': ['Home Decor', 'Side Tables', 'Industrial'], 'user_reviews': [{'review_date': '2024-02-20', 'rating': 4.8, 'comme

In [46]:
response.usage_metadata['input_tokens']

396

In [2]:
from langchain_community.embeddings import OllamaEmbeddings

# Use the model name as per your Ollama configuration
embedding_model = OllamaEmbeddings(model='mxbai-embed-large')

# Get embeddings for a sample text
embedding_vector = embedding_model.embed_query("Sample text to embed")


In [3]:
embedding_vector

[-0.0999850183725357,
 -0.15549291670322418,
 0.5932600498199463,
 -0.4387322664260864,
 -0.17617209255695343,
 -0.39379510283470154,
 -0.03064412623643875,
 -0.06867928802967072,
 0.8101797103881836,
 0.038844432681798935,
 0.050076939165592194,
 -0.057808369398117065,
 0.35805171728134155,
 0.3518208861351013,
 0.16076809167861938,
 -0.23220939934253693,
 0.2587404251098633,
 -0.9445016384124756,
 -0.28887268900871277,
 -0.44790971279144287,
 -1.3317173719406128,
 0.35765889286994934,
 -1.5118088722229004,
 0.004901222884654999,
 -0.10600347071886063,
 0.30414992570877075,
 -0.3531216084957123,
 -0.014453679323196411,
 0.8766957521438599,
 0.18040093779563904,
 -0.3659437596797943,
 0.7301799654960632,
 -0.06891830265522003,
 -0.775624692440033,
 -0.11701775342226028,
 -0.12896504998207092,
 0.30855607986450195,
 -0.8780799508094788,
 -0.4348897635936737,
 -0.11561395227909088,
 0.05516835302114487,
 -0.16026687622070312,
 0.8843611478805542,
 -0.9932461977005005,
 -0.751385748386383

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0,
    # other params...
)

In [3]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

ResponseError: model "llama3.1" not found, try pulling it first (status code: 404)

In [2]:
from langchain_community.llms.ollama import Ollama

# Load the local Ollama model (llama3:8b)
llm = Ollama(model="deepseek-coder:6.7b")

# Generate a simple response
response = llm.invoke("What is LangChain and how does it work?")
print(response)


KeyboardInterrupt: 